In [2]:
import struct
import math
from typing import Union

In [3]:
import struct
import math
from typing import Union, Tuple

def f32_to_u32(x: float) -> int:
    """float -> uint32 (IEEE-754 fp32 bitpattern)"""
    b = struct.pack(">f", float(x))
    return struct.unpack(">I", b)[0]

def u32_to_f32(u: int) -> float:
    """uint32 bitpattern -> float (IEEE-754 fp32)"""
    b = struct.pack(">I", u & 0xFFFFFFFF)
    return struct.unpack(">f", b)[0]

def f32_to_hex(x: float) -> str:
    return f"0x{f32_to_u32(x):08x}"

def hex_to_u32(h: Union[str, int]) -> int:
    if isinstance(h, int):
        return h & 0xFFFFFFFF
    s = h.strip().lower()
    if s.startswith("0x"):
        s = s[2:]
    return int(s, 16) & 0xFFFFFFFF

def hex_to_f32(h: Union[str, int]) -> float:
    return u32_to_f32(hex_to_u32(h))

def quantize_fp32(x: float) -> Tuple[float, str]:
    """
    强制把 python float 舍入/量化到 fp32：
      x -> bits -> fp32_float
    返回 (fp32_float, hex_string)
    """
    u = f32_to_u32(x)
    return u32_to_f32(u), f"0x{u:08x}"

def print_val(tag: str, x: float) -> Tuple[float, str]:
    xf, hx = quantize_fp32(x)
    print(f"{tag}: float={xf!r}  hex={hx}")
    return xf, hx

def fp32_op(a: float, b: float, op: str) -> float:
    # 注意：a,b 已经是 fp32 值（由 quantize_fp32 得到）
    if op == "+":
        return a + b
    if op == "-":
        return a - b
    if op == "*":
        return a * b
    if op == "/":
        return a / b
    raise ValueError(op)

def test_one_pair(a_in: float, b_in: float) -> None:
    print("====== NEW CASE ======")

    # 1) 输入先做 fp32 量化，并打印输入 float/hex
    a_fp32, a_hex = print_val("A_in(fp32)", a_in)
    b_fp32, b_hex = print_val("B_in(fp32)", b_in)

    # 2) 用量化后的输入做运算；结果再量化到 fp32，并打印 float/hex
    for op in ["+", "-", "*", "/"]:
        try:
            r = fp32_op(a_fp32, b_fp32, op)
        except Exception as e:
            print(f"A {op} B: EXCEPTION {e}")
            continue

        r_fp32, r_hex = quantize_fp32(r)
        print(f"A {op} B: float={r_fp32!r}  hex={r_hex}")

    print()

def main():
    # 你可以把这里换成你 Verilog testbench 的激励向量
    cases = [
        (1.5, 0.75),
        (3.1415926, -2.0),
        (1e20, 1e-10),
        (1.0, 0.0),          # divide by zero -> inf
        (-0.0, 0.0),         # signed zero
        (float("nan"), 1.0), # NaN propagation
    ]
    for a, b in cases:
        test_one_pair(a, b)

    # 额外：演示 hex -> float（如果你需要从波形里抄 hex 回来对）
    print("====== HEX DECODE DEMO ======")
    for h in ["0x3f800000", "0xbf800000", "0x7f800000", "0x7fc00000"]:
        x = hex_to_f32(h)
        print(f"hex={h}  float={x!r}")

if __name__ == "__main__":
    main()

====== NEW CASE ======
A_in(fp32): float=1.5  hex=0x3fc00000
B_in(fp32): float=0.75  hex=0x3f400000
A + B: float=2.25  hex=0x40100000
A - B: float=0.75  hex=0x3f400000
A * B: float=1.125  hex=0x3f900000
A / B: float=2.0  hex=0x40000000

====== NEW CASE ======
A_in(fp32): float=3.141592502593994  hex=0x40490fda
B_in(fp32): float=-2.0  hex=0xc0000000
A + B: float=1.1415925025939941  hex=0x3f921fb4
A - B: float=5.141592502593994  hex=0x40a487ed
A * B: float=-6.283185005187988  hex=0xc0c90fda
A / B: float=-1.570796251296997  hex=0xbfc90fda

====== NEW CASE ======
A_in(fp32): float=1.0000000200408773e+20  hex=0x60ad78ec
B_in(fp32): float=1.000000013351432e-10  hex=0x2edbe6ff
A + B: float=1.0000000200408773e+20  hex=0x60ad78ec
A - B: float=1.0000000200408773e+20  hex=0x60ad78ec
A * B: float=10000000000.0  hex=0x501502f9
A / B: float=1.0000000150474662e+30  hex=0x7149f2ca

====== NEW CASE ======
A_in(fp32): float=1.0  hex=0x3f800000
B_in(fp32): float=0.0  hex=0x00000000
A + B: float=1.0  hex=

In [8]:
print(hex_to_f32("3fc00000"))
hex_to_f32("3f2d1768")

1.5


0.6761384010314941

In [3]:
import struct
from typing import Tuple

def f32_to_u32(x: float) -> int:
    b = struct.pack(">f", float(x))
    return struct.unpack(">I", b)[0]

def u32_to_f32(u: int) -> float:
    b = struct.pack(">I", u & 0xFFFFFFFF)
    return struct.unpack(">f", b)[0]

def quantize_fp32(x: float) -> Tuple[float, str]:
    """x -> fp32 bits -> fp32 float ; return (fp32_float, hex)"""
    u = f32_to_u32(x)
    return u32_to_f32(u), f"0x{u:08x}"

def print_fp32(tag: str, x: float) -> float:
    xf, hx = quantize_fp32(x)
    print(f"{tag}: float={xf!r}  hex={hx}")
    return xf

def fp32_add(a: float, b: float) -> float:
    # a,b assumed already fp32 values
    r = a + b
    return quantize_fp32(r)[0]  # fp32 writeback

def fp32_sub(a: float, b: float) -> float:
    r = a - b
    return quantize_fp32(r)[0]

def fp32_mul(a: float, b: float) -> float:
    r = a * b
    return quantize_fp32(r)[0]

def test_lerp(y0_in: float, y1_in: float, t_in: float) -> None:
    print("====== LERP CASE: golden = y0 + (y1 - y0) * t ======")

    # 1) inputs quantized to fp32 (as hardware would see them)
    y0 = print_fp32("y0_in(fp32)", y0_in)
    y1 = print_fp32("y1_in(fp32)", y1_in)
    t  = print_fp32("t_in (fp32)", t_in)

    # 2) step-by-step fp32 ops (each stage writeback quantized)
    diff = fp32_sub(y1, y0)
    print_fp32("diff = y1 - y0 (fp32)", diff)

    prod = fp32_mul(diff, t)
    print_fp32("prod = diff * t (fp32)", prod)

    golden = fp32_add(y0, prod)
    print_fp32("golden = y0 + prod (fp32)", golden)

    print()

def main():
    cases = [
        (0.0, 1.0, 0.5),
        (1.5, 0.75, 0.25),
        (3.1415926, -2.0, 0.1),
        (1e20, 1e20 + 1.0, 0.5),  # 精度差异示例（y1-y0 可能变 0）
        (-0.0, 0.0, 0.3),
    ]
    for y0, y1, t in cases:
        test_lerp(y0, y1, t)

if __name__ == "__main__":
    main()

====== LERP CASE: golden = y0 + (y1 - y0) * t ======
y0_in(fp32): float=0.0  hex=0x00000000
y1_in(fp32): float=1.0  hex=0x3f800000
t_in (fp32): float=0.5  hex=0x3f000000
diff = y1 - y0 (fp32): float=1.0  hex=0x3f800000
prod = diff * t (fp32): float=0.5  hex=0x3f000000
golden = y0 + prod (fp32): float=0.5  hex=0x3f000000

====== LERP CASE: golden = y0 + (y1 - y0) * t ======
y0_in(fp32): float=1.5  hex=0x3fc00000
y1_in(fp32): float=0.75  hex=0x3f400000
t_in (fp32): float=0.25  hex=0x3e800000
diff = y1 - y0 (fp32): float=-0.75  hex=0xbf400000
prod = diff * t (fp32): float=-0.1875  hex=0xbe400000
golden = y0 + prod (fp32): float=1.3125  hex=0x3fa80000

====== LERP CASE: golden = y0 + (y1 - y0) * t ======
y0_in(fp32): float=3.141592502593994  hex=0x40490fda
y1_in(fp32): float=-2.0  hex=0xc0000000
t_in (fp32): float=0.10000000149011612  hex=0x3dcccccd
diff = y1 - y0 (fp32): float=-5.141592502593994  hex=0xc0a487ed
prod = diff * t (fp32): float=-0.5141592621803284  hex=0xbf039ff1
golden = y0 

In [4]:
import struct
from typing import Union, Tuple, Optional

def hex_to_u32(h: Union[str, int]) -> int:
    if isinstance(h, int):
        return h & 0xFFFFFFFF
    s = h.strip().lower()
    if s.startswith("0x"):
        s = s[2:]
    return int(s, 16) & 0xFFFFFFFF

def u32_to_hex(u: int) -> str:
    return f"{u & 0xFFFFFFFF:08x}"

def u32_to_f32(u: int) -> float:
    b = struct.pack(">I", u & 0xFFFFFFFF)
    return struct.unpack(">f", b)[0]

def f32_to_u32(x: float) -> int:
    b = struct.pack(">f", float(x))
    return struct.unpack(">I", b)[0]

def quantize_fp32(x: float) -> float:
    """python float -> fp32 -> python float (simulating fp32 writeback)"""
    return u32_to_f32(f32_to_u32(x))

def print_from_hex(tag: str, hx: Union[str, int]) -> float:
    u = hex_to_u32(hx)
    f = u32_to_f32(u)
    print(f"{tag}: hex={u32_to_hex(u)}  float={f!r}")
    return f

def print_fp32(tag: str, x: float) -> None:
    u = f32_to_u32(x)
    print(f"{tag}: hex={u32_to_hex(u)}  float={u32_to_f32(u)!r}")

def fp32_add(a: float, b: float) -> float:
    return quantize_fp32(a + b)

def fp32_sub(a: float, b: float) -> float:
    return quantize_fp32(a - b)

def fp32_mul(a: float, b: float) -> float:
    return quantize_fp32(a * b)

def lerp_from_hex(y0_hex: Union[str, int],
                  y1_hex: Union[str, int],
                  t_hex: Union[str, int],
                  got_hex: Optional[Union[str, int]] = None) -> Tuple[str, float]:
    """
    输入：y0/y1/t 的 fp32 hex（如 '00000000', '40800000', '3f000000'）
    输出：golden 的 (hex_str, float_val)，并打印全过程
    """
    print("====== LERP from HEX: golden = y0 + (y1 - y0) * t ======")

    # 1) decode inputs
    y0 = print_from_hex("y0", y0_hex)
    y1 = print_from_hex("y1", y1_hex)
    t  = print_from_hex("t ", t_hex)

    # 2) fp32 pipeline (writeback each stage)
    diff = fp32_sub(y1, y0)
    print_fp32("diff = y1 - y0", diff)

    prod = fp32_mul(diff, t)
    print_fp32("prod = diff * t", prod)

    golden = fp32_add(y0, prod)
    
    u = f32_to_u32(golden)
    golden_hex = u32_to_hex(u)
    print("golden: {golden_hex}")
    print_fp32("golden", golden)

    # 3) optional: compare to got_hex
    if got_hex is not None:
        got_u = hex_to_u32(got_hex)
        got_f = u32_to_f32(got_u)
        ok = (got_u == u)
        print(f"got   : hex={u32_to_hex(got_u)}  float={got_f!r}  => {'MATCH' if ok else 'MISMATCH'}")

    print()
    return golden_hex, golden

def main():
    # 你的例子：
    lerp_from_hex("00000000", "40800000", "3f000000", got_hex="40000000")

    # 你也可以加更多 case
    lerp_from_hex("3f800000", "40400000", "3e800000")  # 1.0 -> 2.0, t=0.5
    lerp_from_hex("bf800000", "3f800000", "3f000000")  # 1.0 -> 2.0, t=0.5

if __name__ == "__main__":
    main()

====== LERP from HEX: golden = y0 + (y1 - y0) * t ======
y0: hex=00000000  float=0.0
y1: hex=40800000  float=4.0
t : hex=3f000000  float=0.5
diff = y1 - y0: hex=40800000  float=4.0
prod = diff * t: hex=40000000  float=2.0
golden: {golden_hex}
golden: hex=40000000  float=2.0
got   : hex=40000000  float=2.0  => MATCH

====== LERP from HEX: golden = y0 + (y1 - y0) * t ======
y0: hex=3f800000  float=1.0
y1: hex=40400000  float=3.0
t : hex=3e800000  float=0.25
diff = y1 - y0: hex=40000000  float=2.0
prod = diff * t: hex=3f000000  float=0.5
golden: {golden_hex}
golden: hex=3fc00000  float=1.5

====== LERP from HEX: golden = y0 + (y1 - y0) * t ======
y0: hex=bf800000  float=-1.0
y1: hex=3f800000  float=1.0
t : hex=3f000000  float=0.5
diff = y1 - y0: hex=40000000  float=2.0
prod = diff * t: hex=3f800000  float=1.0
golden: {golden_hex}
golden: hex=00000000  float=0.0

